# VL10 – Mask language models, BERT, and applications

In the previous lab we **implemented a decoder-only Transformer** from scratch.

In this lab we switch gears and **use a pretrained encoder-only Transformer (BERT)** from HuggingFace.
We will:

1. Load a pretrained **BERT encoder** and explore tokenization and hidden states.
2. Try **masked language modeling (MLM)**.
3. Run a **tiny fine-tuning step** of BERT on IMDB sentiment classification (to see the mechanics).

## 1. Imports and device setup

We import:

- `transformers` for pretrained BERT models and tokenizer.
- `datasets` for a small IMDB sample.
- `torch` for tensors and optimization.

We also detect whether a GPU is available.

In [ ]:
import torch
import numpy as np

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. Loading a pretrained BERT model

There are multiple BERT-family models available on HuggingFace.  
They differ mainly in size, speed, and accuracy:

- **bert-base-uncased:** ~450 MB (official model)
- **distilbert-base-uncased:** ~250 MB (distilled, faster)
- **prajjwal1/bert-mini:** ~45 MB (small, good for teaching)
- **prajjwal1/bert-tiny:** ~20 MB (very small, fastest)

During class we will use a *smaller* model due to time and infrastructure constraints.


### Downloading models on systems **without internet access in Jupyter**

Our environment does not allow internet downloads inside notebooks.  
We therefore download models **in the terminal** using a helper script:

```bash
$ python scripts/download_models.py "prajjwal1/bert-mini" models/prajjwal1_bert-mini
```
This saves the entire model locally under models/prajjwal1_bert-mini.

Once downloaded, we can load it offline from Jupyter by pointing to the local directory.


Now that we have chosen a model, we should download two components:

- `AutoTokenizer`: loads the tokenizer associated with the selected checkpoint  
  (for BERT models, usually WordPiece tokenization).

- `AutoModel`: loads the base model architecture associated with the checkpoint  
  (for BERT checkpoints, an encoder-only Transformer that outputs hidden states).

These automatically choose the correct model architecture from the local folder.

In [ ]:
from transformers import AutoTokenizer, AutoModel

## If you have Internet, you can use these models
#tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
#bert_encoder = BertModel.from_pretrained("bert-base-uncased").to(device)

# If you are using the non official bert models

tokenizer = AutoTokenizer.from_pretrained("../../models/prajjwal1_bert-mini")
bert_encoder = AutoModel.from_pretrained("../../models/prajjwal1_bert-mini").to(device)

## 3. Tokenizing simple sentences

We pass a short sentence to the tokenizer and inspect the result.

The tokenizer:

- adds special tokens `[CLS]` and `[SEP]`,
- converts tokens to integer IDs,
- creates attention masks, and
- (optionally) token type IDs (segment embeddings).

In [ ]:
text_a = "Transformers are amazing."
text_b = "They are fantastic."

encoded = tokenizer(
    text=text_a,
    text_pair=text_b,
    return_tensors="pt"
)

# Let's extract the token ids and then convert it back to tokens 
# to see what the encoding has done
token_ids = encoded["input_ids"][0].tolist()
print(tokenizer.convert_ids_to_tokens(token_ids))
print("token_ids     :", encoded["input_ids"][0].tolist())
print("token_type_ids:", encoded["token_type_ids"][0].tolist())
print("attention_mask:", encoded["attention_mask"][0].tolist())

encoded

## 4. Forward pass: From token IDs to contextual embeddings

Now that we have tokenized our input, the next step is to feed it into BERT.

BERT takes:
- **input_ids**: the numerical WordPiece tokens  
- **token_type_ids**: segment A/B indicators (all zeros for single sentences)  
- **attention_mask**: tells BERT which tokens are real (1) and which are padding (0)

What we get back from the model is a set of **contextual embeddings**:

- A vector for each token in the sequence  
- Plus a special vector for the **`[CLS]` token**, which summarizes the entire sequence  
  (this is used later for sequence-level classification tasks)

Let’s run a forward pass and look at these embeddings.

In [ ]:
# Move inputs to the device (CPU or GPU)
encoded = {k: v.to(device) for k, v in encoded.items()}

# Forward pass through the encoder (no gradient tracking needed here)
with torch.no_grad():
    outputs = bert_encoder(**encoded)

# Extract the hidden states for each token
last_hidden = outputs.last_hidden_state
print("Hidden state shape:", last_hidden.shape)

# The [CLS] embedding is always the first token (index 0) [B, T, C]
cls_embedding = last_hidden[:, 0, :]
print("CLS embedding shape:", cls_embedding.shape)

## 5. Masked Language Modeling (MLM)

In the lecture we saw that BERT is trained with **Masked Language Modeling (MLM)**:

- We **mask** some tokens with `[MASK]`.
- The model has to **predict the original word**, using *both left and right context*.
- This is what makes BERT **bidirectional**.

Our smaller `bert-mini` model was pretrained with the same idea.  
Here we load a **Masked LM head** on top of the encoder and let it “fill in the blank” in a simple sentence.

In [ ]:
from transformers import AutoModelForMaskedLM

# If you are offline and have downloaded the model locally:
mlm_model = AutoModelForMaskedLM.from_pretrained("../../models/prajjwal1_bert-mini").to(device)
mlm_model.eval()

def predict_mask(text, tokenizer, mlm_model, top_k=5):
    """
    Given a sentence containing [MASK], return the top_k predicted tokens
    from a Masked Language Model.
    """

    if "[MASK]" not in text:
        raise ValueError("Your input sentence must contain the [MASK] token.")

    print(f"\nInput: {text}")

    # Tokenize and move to device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(mlm_model.device) for k, v in inputs.items()}

    # Forward pass
    with torch.no_grad():
        outputs = mlm_model(**inputs)
        logits = outputs.logits

    # Locate the [MASK] token
    # Step 1: compare every token id with tokenizer.mask_token_id -> True at the masked position
    # Step 2: nonzero(as_tuple=True) returns the coordinates of all True values:
    #         (batch_indices, token_positions)
    # Step 3: [1] keeps only the token position(s) within the sequence
    mask_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    mask_logits = logits[0, mask_index, :]

    # Top-k predictions
    top_tokens = torch.topk(mask_logits, k=top_k, dim=-1)

    token_ids = top_tokens.indices[0].tolist()
    probs = top_tokens.values[0].softmax(dim=-1).tolist()

    print("Top predictions:")
    for tid, p in zip(token_ids, probs):
        token = tokenizer.decode([tid])
        print(f"  {token:>10s}  --  {p:.3f}")

    return token_ids

In [ ]:
# Does it encode world knowledge?
predict_mask("The united states is a [MASK].", tokenizer, mlm_model)
predict_mask("Paris is the capital of [MASK].", tokenizer, mlm_model)

# Does it learn grammar?
predict_mask("The child was [MASK] the show on the TV.", tokenizer, mlm_model)
predict_mask("She has [MASK] to school.", tokenizer, mlm_model)

# Does it learn logical structures?
predict_mask("I wanted to go, [MASK] it started raining.", tokenizer, mlm_model)
predict_mask("He studied hard, [MASK] he still failed the exam.", tokenizer, mlm_model)

# Does it use pragmatic context to predict sentiment?
predict_mask("The restaurant was great, the food tasted [MASK].", tokenizer, mlm_model)
predict_mask("The restaurant was really awful, the food tasted [MASK].", tokenizer, mlm_model)

# Coreference resolution
predict_mask("The girl didn’t cross the road because [MASK] was too tired.", tokenizer, mlm_model)
predict_mask("The girl didn’t cross the road because [MASK] was too too wide.", tokenizer, mlm_model)

### Reflection
1. Which types of knowledge does BERT seem to capture well (facts, syntax, sentiment, commonsense)?
2. Which predictions surprised you or seemed incorrect?
3. Why do you think predicting a masked token requires both left and right context?

## 6. Contextual Representation
One of the key use cases of BERT is contextual representations. This code defines a helper function, `get_token_embedding`, which returns the contextual embedding of a target word inside a sentence using a BERT-style model.

Using this code, we can obtain the representation from the last layer, or by averaging nlayers.

In [ ]:

def get_token_embedding(sentence, target_word, tokenizer, model, nlayers=1):
    """
    Extract the contextual embedding of `target_word` in `sentence`.

    Returns:
        (embedding_vector, token_text, token_index)
    """
    # Tokenize
    encoded = tokenizer(sentence, return_tensors="pt")
    input_ids = encoded["input_ids"][0]

    # Convert token ids to string tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    # Find the token index (WordPiece!) that matches the target word
    # If the word splits (e.g. 'banking' → 'bank', '##ing'), we match the first piece.
    target_pieces = tokenizer.tokenize(target_word)
    first_piece = target_pieces[0]

    try:
        target_idx = tokens.index(first_piece)
    except ValueError:
        raise ValueError(f"Could not find token {target_word!r} in: {tokens}")

    # Run BERT
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        if (nlayers==1):
            outputs = model(**encoded)
            hidden = outputs.last_hidden_state  # (batch, seq, hidden_dim)
            embedding = hidden[0, target_idx].cpu()
        else:
            outputs = model(**encoded, output_hidden_states=True)
            hidden_states = outputs.hidden_states

            if nlayers > len(hidden_states):
                raise ValueError(f"Model only has {len(hidden_states)} layers; nlayers={nlayers} is too large.")            
            
            selected = hidden_states[-nlayers:]
            stacked = torch.stack(selected, dim=0)    # (nlayers, batch, seq, dim)
            combined = stacked.mean(dim=0)            # (batch, seq, dim)
            embedding = combined[0, target_idx].cpu() # (dim)
            
    return embedding, tokens[target_idx], target_idx


After defning this function we can try to see if the meanings of bank are indeed clustered together, depending on the sense for finance or river.

In [ ]:
sentences_finance = [
    "He deposited cash in the bank yesterday.",
    "The bank approved her loan application.",
    "They opened a new account at the bank.",
]

sentences_river = [
    "She sat on the river bank and watched the water.",
    "The fisherman waited quietly on the river bank.",
    "The river overflowed its bank after the storm.",
]

all_sentences = sentences_finance + sentences_river
labels = ["finance"] * len(sentences_finance) + ["river"] * len(sentences_river)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

embeddings = []
for s in all_sentences:
    emb, token, idx = get_token_embedding(s, "bank", tokenizer, bert_encoder, nlayers=2)
    embeddings.append(emb.numpy())


X = np.vstack(embeddings)
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

plt.figure(figsize=(6, 6))

for label, (x, y) in zip(labels, coords):
    color = "blue" if label == "finance" else "green"
    plt.scatter(x, y, color=color)
    plt.text(x + 0.01, y + 0.01, label, fontsize=9)

plt.title("Contextual Embeddings of 'bank' (bert-mini)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()

### Sentence Similarity with BERT

We compare two common ways to turn BERT’s token outputs into a single sentence embedding:

**CLS Embedding**
Use the `[CLS]` token from the final (or last 4 averaged) layers. This is simple and fast, but can be unstable semantically, especially for small models.

**Mean Pooling**
Average all token embeddings in the sentence (ignoring padding). It is more stable and usually more meaningful for semantic similarity. Often performs better than CLS in practice.

Both methods produce a sentence vector, and we compute similarity using cosine similarity.

In [ ]:
import torch
import torch.nn.functional as F

# --------------------------------
# Similarity Functions
# --------------------------------

def embed_sentence(sentence, tokenizer, model, method="cls", nlayers=4):
    """
    Return a sentence embedding using either CLS or mean pooling.
    """
    enc = tokenizer(sentence, return_tensors="pt", truncation=True).to(model.device)

    with torch.no_grad():
        out = model(**enc, output_hidden_states=True)
        hidden_states = out.hidden_states  # tuple: layer0 ... layerN

        # Average last nlayers
        stacked = torch.stack(hidden_states[-nlayers:])  # (nlayers, batch, seq, hidden)
        combined = stacked.mean(dim=0)[0]  # (seq, hidden_dim)

    if method == "cls":
        # CLS is at position 0
        return combined[0]
    elif method == "mean":
        # Mean over tokens (excluding padding)
        attention_mask = enc["attention_mask"][0].unsqueeze(1)  # (seq, 1)
        valid_tokens = combined * attention_mask
        return valid_tokens.sum(dim=0) / attention_mask.sum()
    else:
        raise ValueError("method must be 'cls' or 'mean'")

def sentence_similarity(s1, s2, tokenizer, model, method="cls", nlayers=4):
    v1 = embed_sentence(s1, tokenizer, model, method, nlayers)
    v2 = embed_sentence(s2, tokenizer, model, method, nlayers)
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()


In [ ]:

similar_pairs = [
    ("The dog is playing in the yard.",
     "A puppy is running outside in the garden."),

    ("The scientist published a new paper.",
     "A researcher wrote an article recently.")
]

different_pairs = [
    ("The cat slept on the warm sofa.",
     "The stock market crashed overnight."),

    ("He fixed the broken computer.",
     "A thunderstorm is forming near the coast."),
    
    # same words, but different meaning
    ("The dog chased the cat.",
     "The cat chased the dog."),

    ("Issue the email address.",
     "Address the email issue."),    
    
    # polysemy: same key word, completely different meaning
    ("He sat down to deposit money in the bank.",
     "He sat down by the bank of the river."),

    # noun vs verb sense
    ("She read a book before bedtime.",
     "She tried to book a flight online."),

    # NEW: plant (factory vs organism)
    ("The workers entered the plant early this morning.",
     "The gardener watered the plant in the afternoon."),
]

# simplify running the comparisons
def test_similarity(pairs, tokenizer, model, description):
    print(f"\n===== {description} =====")
    for s1, s2 in pairs:
        sim_cls  = sentence_similarity(s1, s2, tokenizer, model, method="cls", nlayers=1)
        sim_mean = sentence_similarity(s1, s2, tokenizer, model, method="mean", nlayers=1)

        print(f"\nSentence 1: {s1}")
        print(f"Sentence 2: {s2}")
        print(f"CLS similarity:       {sim_cls:.3f}")
        print(f"Mean-pooling similarity: {sim_mean:.3f}")


# Run both sets
test_similarity(similar_pairs, tokenizer, bert_encoder, "EXPECTED HIGH SIMILARITY")
test_similarity(different_pairs, tokenizer, bert_encoder, "EXPECTED LOW SIMILARITY")

## 7. Mini fine-tuning of BERT for IMDB sentiment classification

Now we briefly **fine-tune BERT** for a supervised task.

- We use `AutoModelForSequenceClassification` with 2 labels.
- We load a **tiny subset** of the IMDB dataset to keep this quick.
- We run just a few optimization steps to illustrate the mechanics.

> ⚠️ This is **not** a full training – just enough to see loss and gradients flow.

### 7.1 Preparing data and pretrained model

In [ ]:
import pandas as pd
from datasets import load_dataset, load_from_disk

#ds = load_dataset("stanfordnlp/imdb")
ds = load_from_disk("../../data/standfordnlp_imdb")

train_full = ds["train"]

split = train_full.train_test_split(
    train_size=1000,
    test_size=300,
    stratify_by_column="label",
    seed=42
)

imdb_train = split["train"]
imdb_val = split["test"]

imdb_train

In [ ]:
max_len = 256 # can go up to 512, which is the seq length of bert-mini

def encode_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_len,
    )

# adds tokenization output
imdb_train_enc = imdb_train.map(encode_batch, batched=True)
imdb_val_enc = imdb_val.map(encode_batch, batched=True)

# HugginFace models expect target to be named 'labels'
imdb_train_enc = imdb_train_enc.rename_column("label", "labels")
imdb_val_enc = imdb_val_enc.rename_column("label", "labels")

imdb_train_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
imdb_val_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

clf_model = AutoModelForSequenceClassification.from_pretrained(
    "../../models/prajjwal1_bert-mini",
    num_labels=2  # e.g., positive vs negative
).to(device)

optimizer = torch.optim.AdamW(clf_model.parameters(), lr=2e-5)

### 7.2 A few training steps

We take small batches from the encoded IMDB subset and:

1. Move them to the device.
2. Compute loss via `AutoModelForSequenceClassification`.
3. Backpropagate and call the optimizer.

We print the loss to see if it decreases.

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

train_loader = DataLoader(imdb_train_enc, batch_size=16, shuffle=True)
val_loader = DataLoader(imdb_val_enc, batch_size=16, shuffle=False)

def evaluate_loss(model, data_loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()

    return total_loss / len(data_loader)


# Training loop
num_epochs = 5

prev_train_loss = None
prev_val_loss = None

for epoch in range(num_epochs):
    clf_model.train()
    total_train_loss = 0.0

    for batch in tqdm(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = clf_model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = evaluate_loss(clf_model, val_loader, device)

    # Compute deltas
    train_delta = (
        avg_train_loss - prev_train_loss
        if prev_train_loss is not None else None
    )
    val_delta = (
        avg_val_loss - prev_val_loss
        if prev_val_loss is not None else None
    )

    # Pretty printing
    train_str = (
        f"{avg_train_loss:.4f}"
        if train_delta is None
        else f"{avg_train_loss:.4f} ({train_delta:+.4f})"
    )
    val_str = (
        f"{avg_val_loss:.4f}"
        if val_delta is None
        else f"{avg_val_loss:.4f} ({val_delta:+.4f})"
    )

    print(f"Epoch {epoch+1}: train loss = {train_str}, val loss = {val_str}")

    prev_train_loss = avg_train_loss
    prev_val_loss = avg_val_loss

### 7.3 Quick inference

We now test the (very slightly) fine-tuned model on two short reviews.
We **do not** expect good accuracy from only 5 steps, but this shows the API.

In [ ]:
def classify_review(text):
    clf_model.eval()
    encoded = tokenizer(text, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = clf_model(**encoded).logits
    probs = logits.softmax(-1).squeeze()
    label_id = probs.argmax().item()
    label = "positive" if label_id == 1 else "negative"
    return label, probs[label_id].item()

test_reviews = [
    "I absolutely loved this film. The acting and story were fantastic.",
    "This was one of the worst movies I have ever seen.",
    "I have seen better movies. I don't understand the hype"
]

for rev in test_reviews:
    lab, conf = classify_review(rev)
    print(f"\nReview: {rev}\nPredicted: {lab} (confidence {conf:.3f})")

## 8. Reflection
1. Does the training loss decrease every epoch?
2. Does the validation loss follow the same trend?
3. If the validation loss starts increasing while training loss decreases, what might be happening?

#### 8.1 Experiments

Hypothesis:
"More data and more training should improve generalisation."

Test this by varying:
- Dataset size
- Number of epochs

Record:
- Final training loss
- Final validation loss

Were your expectations met?